<p align="center" width="100%">
    <img width="40%" src="customer_support_icon.JPG"> 
</p>

A retail company is on a transformative journey, aiming to elevate their customer services through cutting-edge advancements in Speech Recognition and Natural Language Processing (NLP). As the machine learning engineer for this initiative, you are tasked with developing functionalities that not only convert customer support audio calls into text but also explore methodologies to extract insights from transcribed texts.

In this dynamic project, we leverage the power of `SpeechRecognition`, `Pydub`, and `spaCy` – three open-source packages that form the backbone of your solution. Your objectives are:
  - Transcribe a sample customer audio call, stored at `sample_customer_call.wav`, to showcase the power of open-source speech recognition technology.
  - Analyze sentiment, identify common named entities, and enhance user experience by searching for the most similar customer calls based on a given query from a subset of their pre-transcribed call data, stored at `customer_call_transcriptions.csv`.

This project is an opportunity to unlock the potential of machine learning to revolutionize customer support. Let's delve into the interplay between technology and service excellence.

In [14]:
!pip install SpeechRecognition
!pip install pydub
!pip install spacy
!python -m spacy download en_core_web_md

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
/bin/bash: python: command not found


In [15]:
# Import required libraries
import pandas as pd

import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

import speech_recognition as sr
from pydub import AudioSegment

import spacy

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/repl/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [16]:
import sys

!{sys.executable} -m spacy download en_core_web_md

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 5.2 MB/s eta 0:00:0000:0100:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [17]:
import spacy

nlp = spacy.load("en_core_web_md")

print("spaCy model loaded successfully!")

spaCy model loaded successfully!


In [18]:

# ==========================================
# 1. Speech Recognition and Audio Properties
# ==========================================

recognizer = sr.Recognizer()

sample_customer_call = sr.AudioFile("sample_customer_call.wav")

with sample_customer_call as source:
    sample_customer_call_data = recognizer.record(source)

transcribed_text = recognizer.recognize_google(
    sample_customer_call_data
)

wav_file = AudioSegment.from_file("sample_customer_call.wav")

frame_rate = wav_file.frame_rate
number_channels = wav_file.channels


# ==========================================
# 2. Sentiment Analysis using VADER
# ==========================================

df = pd.read_csv("customer_call_transcriptions.csv")

sid = SentimentIntensityAnalyzer()

df["polarity_score_compound"] = df["text"].apply(
    lambda x: sid.polarity_scores(x)["compound"]
)

def classify(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["polarity_score_compound"].apply(classify)

true_positive = int(
    (
        (df["sentiment_label"] == "positive") &
        (df["sentiment"] == "positive")
    ).sum()
)


# ==========================================
# 3. Named Entity Recognition using spaCy
# ==========================================

nlp = spacy.load("en_core_web_md")

entity_dict = {}

for text in df["text"]:
    doc = nlp(text)

    for ent in doc.ents:
        if ent.text in entity_dict:
            entity_dict[ent.text] += 1
        else:
            entity_dict[ent.text] = 1

most_freq_ent = max(entity_dict, key=entity_dict.get)


# ==========================================
# 4. Find Most Similar Customer Call
# ==========================================

input_query = "wrong package delivery"

processed_query = nlp(input_query)

df["similarity"] = df["text"].apply(
    lambda x: processed_query.similarity(nlp(x))
)

most_similar_text = df.loc[
    df["similarity"].idxmax(), "text"
]


# ==========================================
# 5. Display Final Results
# ==========================================

print("Transcribed text:", transcribed_text)
print("Frame rate:", frame_rate)
print("Number of channels:", number_channels)
print("True positive:", true_positive)
print("Most frequent entity:", most_freq_ent)
print("Most similar text:", most_similar_text)


Transcribed text: hello I'm experiencing an issue with your product I'd like to speak to someone about a replacement
Frame rate: 44100
Number of channels: 1
True positive: 2
Most frequent entity: yesterday
Most similar text: wrong package delivered
